## **Data Science Aplicado a las Finanzas** 🚀
### **HW Sesiones 9 - 10**

Andrés C. Medina Sanhueza

Senior Datascientist Engineer 

anmedinas@gmail.com

Consideraciones Previas

* Tarea es Individual, debe subir al repositorio personal (notebooks)

* Fecha Entrega : Viernes 31 de Julio 23:59 (se recomienda hacer `commits` cada vez que hagan cambios o subir, esto, también es parte de la evaluación, es decir, trazabilidad del trabajo).

---

## 1. Aprendizaje no Supervisado — Segmentación de Clientes para Otorgamiento de Tarjetas de Crédito

Considere el dataset `creditSIM.xlsx` que contiene información de clientes de un banco. El objetivo es construir perfiles o *clusters* de clientes de modo que, mediante estrategias de campañas de marketing, se pueda decidir a quién otorgar tarjetas de crédito. El dataset contiene las siguientes variables:

1. `ID`: Número de cliente.
2. `AgnosDirec`: Años que el cliente lleva viviendo en la misma dirección.
3. `AgnosEmpleo`: Años de antigüedad del cliente en el empleo.
4. `DeudaExt`: Deuda del cliente externa al banco.
5. `DeudaInt`: Deuda del cliente en el banco (por ejemplo, otros créditos).
6. `Edad`: Edad del cliente.
7. `Ingreso`/`Ingreso2`: Ingreso del cliente (recopilados de dos fuentes diferentes).
8. `Nacionalidad`: Nacionalidad del cliente.
9. `NivelEdu`: Nivel educacional del cliente.
10. `VarObj`: Variable objetivo — si el cliente cae en bancarrota al recibir el crédito (`S`) o no (`N`).

⚠️ **Regla del ejercicio:** `VarObj` **no puede** usarse como *input* del proceso de clustering (sería *leakage*, ya que en producción no se conoce el comportamiento futuro del cliente al momento de aprobar la tarjeta). Su único uso permitido es como variable de **validación externa** en la Sección 1.6.

### 1.1 Auditoría y Preprocesamiento de Datos

**(a)** Reporte tipos de datos, cardinalidad de las variables categóricas, porcentaje de valores nulos por columna y estadística descriptiva de las variables numéricas. Justifique una estrategia de imputación (o eliminación) para los nulos encontrados — no use `dropna()` sin antes cuantificar cuánta información se pierde.

**(b)** Analice `Ingreso` e `Ingreso2`: ¿son consistentes entre sí (correlación, diferencias sistemáticas)? Decida y justifique si usará una, ambas, un promedio, o una variable derivada (por ejemplo, la discrepancia entre ambas como *proxy* de riesgo de fraude declarativo).

**(c)** Identifique outliers univariados en las variables continuas (`DeudaExt`, `DeudaInt`, `Ingreso*`, `Edad`, `AgnosEmpleo`) usando al menos dos criterios (por ejemplo, IQR y z-score robusto con MAD). Decida si los trata (winsorización, transformación logarítmica) o los conserva, justificando el impacto que tendrían sobre una distancia euclidiana no tratada.

**(d)** Construya al menos **dos variables derivadas** con sentido de negocio (por ejemplo, `Deuda_Total/Ingreso` como *leverage ratio*, o `DeudaInt/DeudaExt`). Justifique por qué podrían mejorar la separación de perfiles de riesgo.

### 1.2 Codificación de Variables Mixtas y Escalamiento

El dataset combina variables numéricas y categóricas (`Nacionalidad`, `NivelEdu`), lo cual invalida el uso directo de una distancia euclidiana estándar.

**(a)** Explique por qué escalar (`StandardScaler`/`RobustScaler`) es indispensable antes de aplicar K-means o PCA, y por qué la elección Standard vs. Robust depende de lo decidido en 1.1(c).

**(b)** Implemente **al menos dos enfoques** para manejar la mezcla de tipos de variables y compare sus resultados en la Sección 1.5:
   - *One-hot encoding* de las categóricas + escalamiento de las numéricas, para usar con K-means/GMM.
   - Distancia de **Gower** (o `k-prototypes`) que maneja variables mixtas nativamente, sin necesidad de one-hot encoding.

### 1.3 Reducción de Dimensionalidad

**(a)** Aplique **PCA** sobre el conjunto de variables numéricas escaladas (incluyendo las derivadas de 1.1(d)). Reporte la varianza explicada acumulada y el número de componentes necesarios para explicar al menos el 80–90%.

**(b)** Grafique el *biplot* de las dos primeras componentes principales, incluyendo los *loadings* de las variables originales. Interprete económicamente qué representa cada componente (por ejemplo, "componente de solvencia/apalancamiento").

**(c)** Analice si las variables categóricas codificadas contaminan la estructura de PCA (dado que son variables 0/1). Comente si sería más apropiado usar **MCA** (*Multiple Correspondence Analysis*) o **FAMD** (*Factor Analysis of Mixed Data*) para este dataset y, si el tiempo lo permite, impleméntelo como alternativa. (averigue en internet sobre estas técnicas alternativas)

### 1.4 Determinación del Número Óptimo de Clusters

Para el pipeline de K-means (sobre datos escalados + codificados):

**(a)** Grafique el método del codo (*inertia* vs. $k$, para $k=2,\dots,10$).

**(b)** Calcule y grafique el **coeficiente de Silhouette promedio** para el mismo rango de $k$.

**(c)** Calcule el índice de **Davies-Bouldin** y el índice de **Calinski-Harabasz** para el mismo rango de $k$.

**(d)** Los tres criterios, ¿coinciden en el $k$ sugerido? Si no coinciden, justifique con criterio de negocio (número de perfiles accionables para una campaña de marketing) cuál $k$ finalmente utilizará.

### 1.5 Comparación de Algoritmos de Clustering

Ajuste **al menos tres algoritmos** distintos con el $k$ (o parámetros equivalentes) seleccionado en 1.4, y sobre el enfoque de codificación mixta elegido en 1.2:

1. **K-means** (o `k-prototypes` si trabajó con distancia de Gower).
2. **Clustering jerárquico aglomerativo** (linkage de Ward), incluyendo el **dendrograma** y la línea de corte utilizada.
3. **DBSCAN** o **HDBSCAN**, comentando la sensibilidad a `eps`/`min_samples` y cuántos clientes quedan etiquetados como ruido.

Compare los cuatro (o tres) resultados usando el índice de Silhouette y el **Adjusted Rand Index (ARI)** entre pares de particiones, para cuantificar qué tan de acuerdo están los algoritmos entre sí.

### 1.6 Perfilamiento de Clusters y Validación Externa

**(a)** Para la partición final elegida, construya una tabla de perfil por cluster con la media/mediana de cada variable original (no transformada) y el tamaño relativo (%) de cada cluster.

**(b)** Nombre cada cluster con una etiqueta de negocio interpretable (por ejemplo, "jóvenes de alto ingreso y bajo apalancamiento", "senior con alta deuda externa").

**(c)** Aunque `VarObj` no se usó para entrenar, calcule la **tasa de bancarrota** (`% de VarObj = 'S'`) dentro de cada cluster. ¿Existen clusters con tasas de default significativamente distintas? Realice un test de proporciones (o $\chi^2$ de independencia) entre `cluster` y `VarObj`. Esto valida si la segmentación no supervisada, sin ver la etiqueta, logra igualmente separar el riesgo de crédito.

**(d)** Evalúe la **estabilidad** de la partición: re-ejecute K-means con al menos 10 semillas distintas (o vía *bootstrap* de las observaciones) y reporte el ARI promedio entre particiones. ¿Es una segmentación robusta o depende fuertemente de la inicialización/muestra?

### 1.7 Detección de Anomalías (Complemento)

Ajuste un modelo de **Isolation Forest** (o *Local Outlier Factor*) sobre las mismas variables para identificar clientes atípicos que podrían representar riesgo de fraude o error de captura, independientemente del cluster al que pertenezcan. Reporte cuántos de esos clientes atípicos tienen `VarObj = 'S'` en comparación con la tasa base de bancarrota del dataset.

### 1.8 Síntesis y Recomendación de Negocio

Redacte (máx. 1 página) una recomendación para el área de marketing/riesgo del banco: qué perfiles priorizar para campañas de tarjetas de crédito, qué perfiles evitar o requieren garantías adicionales, y qué limitaciones tiene esta segmentación (por ejemplo, ausencia de variables de comportamiento transaccional, estabilidad temporal de los clusters, riesgo de sesgos si `Nacionalidad` correlaciona con variables protegidas).</cell id="37120e06">


In [ ]:
# inserte aqui su codigo

## 2. Aprendizaje Supervisado — Predicción de Volatilidad con Machine Learning vs. Modelos GARCH

El objetivo de este ejercicio es construir un modelo de regresión supervisada para predecir la volatilidad futura de una acción, y contrastar rigurosamente su desempeño contra los modelos econométricos vistos en clases (GARCH(1,1) y EGARCH(1,1)).

### 2.1 Descarga y Construcción de Variables

**(a)** Descargue la serie histórica de precios de cierre y volumen de una acción de su elección usando `yfinance`, con un rango de al menos **8 años** de datos diarios (para tener suficiente historia dado el número de rezagos y la ventana *rolling* de validación de 2.5).

**(b)** Construya las siguientes variables:

   - **Retorno logarítmico diario:** $r_t = \log(\text{Close}_t / \text{Close}_{t-1})$.
   - **Variable objetivo — volatilidad realizada futura:** $\text{RV}_{t}$, la desviación estándar de $r$ en una ventana móvil de 30 días **hacia adelante** (es decir, el modelo debe predecir volatilidad *futura*, no simplemente replicar la volatilidad *pasada* con la que se construyen los *features*; sea explícito en el desfase temporal usado y evite *look-ahead bias*).
   - **Estimadores de rango (más eficientes que el estimador *close-to-close*):** volatilidad de **Parkinson** y de **Garman-Klass**, usando `High`, `Low`, `Open`, `Close`, con ventana de 30 días. Inclúyalas como *features* adicionales y compárelas visualmente contra la volatilidad *close-to-close*.
   - **Cambio porcentual de volumen:** $\text{Vol\_change}_t = (\text{Volume}_t - \text{Volume}_{t-1}) / \text{Volume}_{t-1}$.
   - **Media móvil de precios** (20 días) y **distancia porcentual del precio respecto a su media móvil** (*proxy* de momentum/sobrecompra).
   - **Skewness y kurtosis móvil** de los retornos (ventana de 30 días) — motivadas por los hechos estilizados de colas pesadas vistos en clases.
   - **Rezagos:** para cada variable anterior, genere al menos **5 rezagos** (`Lag_Return_1`, ..., `Lag_Return_5`, etc.).
   - **Volatilidad implícita (VIX):** descargue la serie histórica del VIX y sincronícela con las fechas de su acción (atención a *holidays*/desfases de calendario entre mercados). Incluya el valor actual y al menos 5 rezagos.
   - **Variables macroeconómicas:** descargue y sincronice al menos **tres** variables macro (por ejemplo, tasa de interés de la Fed, inflación, tasa de desempleo, spread de tasas 10Y-2Y) desde `FRED` (vía `pandas_datareader` o `fredapi`), generando al menos 5 rezagos de cada una. Dado que estas series suelen publicarse con menor frecuencia (mensual), documente explícitamente el método de sincronización usado (`ffill`/`asof`) y por qué evita *look-ahead bias*.

### 2.2 Análisis Exploratorio y Diagnóstico de Estacionariedad

**(a)** Aplique el test **ADF (Augmented Dickey-Fuller)** a la variable objetivo (`RV_t`) y a las variables macro utilizadas. ¿Alguna requiere diferenciación para ser estacionaria? Justifique si la usa en niveles, en diferencias, o en ambas formas como *features* separados.

**(b)** Grafique la matriz de correlación entre todos los *features* (o al menos entre las variables sin rezagar) y comente sobre potenciales problemas de **multicolinealidad** entre los rezagos de una misma variable — relevante especialmente para SVR, menos crítico para Random Forest.

### 2.3 Preparación de Datos

**(a)** Elimine los valores nulos generados por los rezagos y ventanas móviles (inicio y fin de la muestra por la ventana *forward*).

**(b)** Divida los datos en entrenamiento/validación/prueba respetando el **orden temporal** (por ejemplo, 70/15/15). Explique por qué un `train_test_split` aleatorio sería metodológicamente incorrecto en este contexto.

**(c)** Escale los *features* usando únicamente estadísticos calculados en el conjunto de entrenamiento (evite *data leakage* del conjunto de prueba hacia el *scaler*).

### 2.4 Entrenamiento, Selección de Hiperparámetros y Modelos

**(a)** Entrene **Random Forest** y **SVR** (los modelos vistos en clases). Para la selección de hiperparámetros use `TimeSeriesSplit` (validación cruzada que respeta el orden temporal) combinado con `GridSearchCV` o `RandomizedSearchCV` — nunca k-fold estándar sobre series de tiempo. Reporte la grilla de hiperparámetros explorada para cada modelo.

**(b)** *(Extensión)* Entrene además un modelo de **Gradient Boosting** (`XGBoost`, `LightGBM` o `GradientBoostingRegressor`) como tercer competidor de *machine learning*.

**(c)** Reporte la **importancia de variables** (impureza para Random Forest/GBM, y *permutation importance* — válida para los tres modelos, incluyendo SVR) de cada modelo ganador. ¿Dominan los rezagos de volatilidad propia, el VIX, o las variables macro? Comente si esto es consistente con la teoría financiera de *volatility clustering*.

### 2.5 Evaluación y Comparación de Desempeño

**(a)** Evalúe cada modelo en el conjunto de prueba usando **RMSE, MAE y QLIKE** — esta última es la función de pérdida estándar en la literatura de *forecasting* de volatilidad, $\text{QLIKE} = \frac{1}{n}\sum_t \left(\frac{\text{RV}_t}{\hat\sigma_t^2} - \ln\frac{\text{RV}_t}{\hat\sigma_t^2} - 1\right)$, y es más robusta a la asimetría de la distribución de la volatilidad que el RMSE.

**(b)** Grafique la volatilidad observada vs. la volatilidad estimada por cada modelo en el conjunto de prueba, en un mismo panel para facilitar la comparación visual.

### 2.6 Benchmark contra Modelos Econométricos

**(a)** Ajuste modelos **GARCH(1,1)** y **EGARCH(1,1)** (vistos en clases) sobre los retornos de la misma acción, y genere pronósticos de volatilidad **fuera de muestra** alineados temporalmente con el mismo período de prueba usado para los modelos de ML (use un esquema *rolling*/*expanding window* explícito y consistente con el usado, si corresponde, en la sección de series de tiempo).

**(b)** Compare RMSE, MAE y QLIKE de los cuatro-cinco modelos (RF, SVR, \[GBM\], GARCH, EGARCH) en una tabla resumen ordenada por desempeño.

**(c)** Aplique el **test de Diebold-Mariano** para determinar si la diferencia de desempeño entre el mejor modelo de ML y el mejor modelo GARCH/EGARCH es **estadísticamente significativa**, o si podría deberse a ruido muestral.

### 2.7 Diagnóstico de Errores

Para el modelo ganador (según 2.6b), analice los errores de predicción $(\text{RV}_t - \hat{\text{RV}}_t)$ en el conjunto de prueba: grafíquelos en el tiempo, evalúe si están correlacionados serialmente (ACF) y si su magnitud se relaciona con períodos de alta volatilidad de mercado (por ejemplo, marcando visualmente eventos de estrés conocidos dentro de su ventana de prueba, si los hay).

### 2.8 Discusión Final

Responda de forma breve y argumentada:

1. ¿El mejor modelo de *machine learning* supera de forma estadísticamente significativa a los benchmarks GARCH/EGARCH? Si no es así, ¿qué tan justificable es, en la práctica, preferir un modelo de caja negra sobre uno econométrico interpretable con desempeño similar?
2. ¿Qué variables resultaron más relevantes según la importancia de *features*, y es consistente con lo que un modelo GARCH asume implícitamente (que el mejor predictor de la volatilidad futura es la volatilidad/los choques pasados)?
3. Mencione al menos **dos limitaciones** del enfoque de ML utilizado para pronóstico de volatilidad financiera (por ejemplo, incapacidad de extrapolar fuera del rango de entrenamiento, ausencia de una dinámica de reversión a la media explícita, riesgo de sobreajuste con el número de *features*/rezagos utilizados) y cómo las abordaría en una extensión futura (por ejemplo, modelos híbridos ML-GARCH, LSTM, o *ensemble* de ambos enfoques).</cell id="4a564cc5">


In [ ]:
# inserte aqui su codigo